In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

POOL_SNAPSHOTS_DIR = Path("../user_data/data/gmx/pool_liquidity/arbitrum/snapshots")
OI_RAW_DIR         = Path("../user_data/data/gmx/open_interest/arbitrum/raw")
SCALE_30           = 10 ** 30

## Load Pool Depth Data

Reads daily LP pool snapshots from `{output_dir}/snapshots/{symbol}/daily.parquet`
produced by `scripts/extract_pool_liquidity.py`.
`pool_tokens` is already converted from raw on-chain amounts using ERC-20 decimals
(e.g. WETH / 1e18, USDC / 1e6).  We sum long + short token pools per market
to get total pool depth per day.  Swap-only markets are excluded.

In [ ]:
frames = []
for symbol_dir in sorted(POOL_SNAPSHOTS_DIR.iterdir()):
    if not symbol_dir.is_dir():
        continue
    for f in symbol_dir.rglob("*.parquet"):
        frames.append(pd.read_parquet(f))

pool = pd.concat(frames, ignore_index=True)
pool["date"] = pd.to_datetime(pool["date"], utc=True)
pool["symbol"] = pool["symbol"].str.replace(r"\s*\[.*\]", "", regex=True).str.strip()

# Drop swap-only markets
pool = pool[~pool["symbol"].str.startswith("SWAP-ONLY")]

pool_daily = pool.groupby(["date", "symbol"])["pool_tokens"].sum().reset_index()
pool_daily.rename(columns={"pool_tokens": "total_pool_tokens"}, inplace=True)

print(f"Markets: {pool_daily['symbol'].nunique()}, "
      f"date range: {pool_daily['date'].min()} \u2192 {pool_daily['date'].max()}")
pool_daily.groupby("symbol")["total_pool_tokens"].mean().sort_values(ascending=False).head(10)

## Load OI Data & Join

In [ ]:
frames = []
for symbol_dir in sorted(OI_RAW_DIR.iterdir()):
    if not symbol_dir.is_dir():
        continue
    for f in symbol_dir.rglob("*.parquet"):
        frames.append(pd.read_parquet(f))

raw = pd.concat(frames, ignore_index=True)
oi = raw[raw["eventType"] == "OpenInterestUpdated"].copy()
oi["ts"] = pd.to_datetime(oi["blockTimestamp"], unit="s", utc=True)
oi["date"] = oi["ts"].dt.normalize()
oi["next_value_usd"] = oi["nextValueUsd"].astype(float) / SCALE_30
oi["symbol"] = oi["symbol"].str.replace(r"\s*\[.*\]", "", regex=True).str.strip()

# Drop swap-only markets
oi = oi[~oi["symbol"].str.startswith("SWAP-ONLY")]

daily_oi = (
    oi.sort_values("ts")
    .groupby(["date", "symbol"])["next_value_usd"]
    .last()
    .reset_index()
    .rename(columns={"next_value_usd": "oi_total_usd"})
)

combined = pool_daily.merge(daily_oi, on=["date", "symbol"], how="inner")
print(f"Combined: {len(combined)} rows, {combined['symbol'].nunique()} markets")
combined.head()

## Chart 1: LP Pool Depth Over Time — Top 10 Markets

Each market is normalised to [0, 1] (min=0, max=1) so WETH-denominated and
USDC-denominated pools can be compared on the same axis.

In [ ]:
top10_pool = (
    pool_daily.groupby("symbol")["total_pool_tokens"]
    .mean()
    .nlargest(10)
    .index.tolist()
)
df_top = pool_daily[pool_daily["symbol"].isin(top10_pool)].copy()

df_top["pool_norm"] = df_top.groupby("symbol")["total_pool_tokens"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)

fig = px.line(
    df_top, x="date", y="pool_norm", color="symbol",
    title="GMX V2 — LP Pool Depth (normalised per market, 0=min, 1=max) — Top 10",
    labels={"pool_norm": "Pool Depth (normalised)", "date": "Date"},
    template="plotly_dark",
)
fig.update_layout(hovermode="x unified")
fig.show()

## Chart 2: OI vs Pool Depth — Dual-Axis Time Series

Left axis = OI in USD &nbsp;|&nbsp; Right axis = pool tokens (human-readable).
Change `MARKET` to explore any symbol.

In [ ]:
MARKET = "ETH/USD"
df_m = combined[combined["symbol"] == MARKET].sort_values("date")

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Scatter(x=df_m["date"], y=df_m["oi_total_usd"],
               name="OI (USD)", line=dict(color="#00cc96")),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=df_m["date"], y=df_m["total_pool_tokens"],
               name="Pool Tokens", line=dict(color="#ab63fa", dash="dash")),
    secondary_y=True,
)
fig.update_layout(
    title=f"{MARKET} — OI (USD) vs LP Pool Depth",
    template="plotly_dark", hovermode="x unified",
)
fig.update_yaxes(title_text="OI (USD)", secondary_y=False)
fig.update_yaxes(title_text="Pool Tokens", secondary_y=True)
fig.show()

## Chart 3: OI vs Pool Depth Scatter — Latest Snapshot

Each bubble = one market.  Bubble size = OI.
X-axis = pool tokens (normalised 0–1 across markets).

**Upper-left** = high OI relative to pool depth (high utilisation — less room for new positions).  
**Lower-right** = deep pools with low OI (most capacity available).

In [ ]:
latest = combined[combined["date"] == combined["date"].max()].copy()
latest["pool_norm"] = latest["total_pool_tokens"] / latest["total_pool_tokens"].max()

fig = px.scatter(
    latest,
    x="pool_norm", y="oi_total_usd",
    text="symbol", size="oi_total_usd", size_max=50,
    title=f"GMX Markets — OI vs Pool Depth ({latest['date'].max().date()})",
    labels={
        "pool_norm": "Pool Depth (normalised, 0–1)",
        "oi_total_usd": "OI (USD)",
    },
    template="plotly_dark",
    color="symbol",
)
fig.update_traces(textposition="top center")
fig.update_layout(showlegend=False)
fig.show()

## Chart 4: OI vs Pool Depth — Overlay Bar (Top 20, sorted by OI)

Green bars = OI (USD). &nbsp;Purple bars = pool depth rescaled to the same axis as OI.

Markets where **green ≈ purple** = high utilisation (pool mostly consumed by open interest).

In [ ]:
latest_sorted = latest.sort_values("oi_total_usd", ascending=False).head(20).copy()
max_oi  = latest_sorted["oi_total_usd"].max()
max_tok = latest_sorted["total_pool_tokens"].max()
latest_sorted["pool_usd_proxy"] = latest_sorted["total_pool_tokens"] / max_tok * max_oi

fig = go.Figure()
fig.add_trace(go.Bar(
    name="OI (USD)",
    x=latest_sorted["symbol"], y=latest_sorted["oi_total_usd"],
    marker_color="#00cc96",
))
fig.add_trace(go.Bar(
    name="Pool depth (scaled to OI axis)",
    x=latest_sorted["symbol"], y=latest_sorted["pool_usd_proxy"],
    marker_color="#636efa", opacity=0.45,
))
fig.update_layout(
    barmode="overlay",
    title="GMX Markets — OI vs Pool Depth (top 20 by OI)",
    template="plotly_dark",
    xaxis_title="Market", yaxis_title="USD (pool normalised to OI scale)",
    xaxis_tickangle=-45,
)
fig.show()

## Universe Output: Sorted by OI, Filtered by Pool Depth

Removes the bottom 25% of markets by pool depth, then sorts the remaining markets
by OI descending.  This is the candidate universe for algorithmic trading.

In [ ]:
threshold = latest["total_pool_tokens"].quantile(0.25)
universe = (
    latest[latest["total_pool_tokens"] > threshold]
    .sort_values("oi_total_usd", ascending=False)
    [["symbol", "oi_total_usd", "total_pool_tokens"]]
    .reset_index(drop=True)
)
universe.columns = ["Symbol", "OI (USD)", "Pool Tokens"]
print("=== GMX Trading Universe (OI-sorted, bottom-25% pool filtered) ===")
universe